In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

Imports loaded.
Database URL created.


# Smoke Test

In [2]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row

('studybook', 'sb_user', datetime.datetime(2026, 5, 7, 23, 28, 10, 747469, tzinfo=datetime.timezone.utc))

# Helper functions to run SQL

In [3]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

In [4]:
# ============================================================
# Cell 4B: Safer Helper Function - Inspect One Table
# Purpose:
# Inspect columns, row count, and sample records for one table.
# ============================================================

def inspect_table_safe(table_name, schema_name="public", sample_size=5):
    """
    Investigate one PostgreSQL table.

    Shows:
    1. Column metadata
    2. Row count
    3. Sample records

    Parameters:
        table_name:   name of the table to inspect
        schema_name:  schema where the table lives, default is public
        sample_size:  number of sample rows to display
    """

    # ------------------------------------------------------------
    # 1. Column metadata
    # ------------------------------------------------------------
    column_query = """
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY
        ordinal_position;
    """

    df_columns = pd.read_sql(
        text(column_query),
        engine,
        params={
            "schema_name": schema_name,
            "table_name": table_name,
        },
    )

    print(f"Column metadata for {schema_name}.{table_name}")
    display(df_columns)

    # ------------------------------------------------------------
    # 2. Row count
    # ------------------------------------------------------------
    count_query = f"""
    SELECT COUNT(*) AS row_count
    FROM {schema_name}.{table_name};
    """

    df_count = pd.read_sql(text(count_query), engine)

    print(f"Row count for {schema_name}.{table_name}")
    display(df_count)

    # ------------------------------------------------------------
    # 3. Sample records
    # ------------------------------------------------------------
    sample_query = f"""
    SELECT *
    FROM {schema_name}.{table_name}
    LIMIT :sample_size;
    """

    df_sample = pd.read_sql(
        text(sample_query),
        engine,
        params={
            "sample_size": sample_size,
        },
    )

    print(f"Sample records from {schema_name}.{table_name}")
    display(df_sample)

# 06 — Server 5-Minute Capacity Rollup Practice

In this section, we practice a realistic capacity-engineering pattern.

The table `server_metric_samples_5min` contains one telemetry sample every 5 minutes for every server.

That means each server has about 12 samples per hour.

The goal is to learn how to roll 5-minute samples into hourly capacity metrics using:

- AVG = normal / typical utilization
- MAX = worst observed spike
- P95 = sustained high pressure, ignoring rare one-off noise

This is the kind of SQL pattern used in telemetry, capacity planning, observability, and performance engineering.

## 06.1 Verify that the practice tables exist

Before writing capacity queries, first verify that the two practice tables were created.

Expected tables:

- server_inventory_practice
- server_metric_samples_5min

The inventory table describes servers.
The metric table stores the 5-minute telemetry samples.

In [5]:
sql = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN (
      'server_inventory_practice',
      'server_metric_samples_5min'
  )
ORDER BY table_name;
"""

run_sql(sql)

,table_schema,table_name
0,public,server_inventory_practice
1,public,server_metric_samples_5min


## 06.2 Inspect the server inventory

This table is the server dimension / lookup table.

It tells us:

- hostname
- service_name
- environment
- region
- CPU capacity
- memory capacity

This is similar to a CMDB or infrastructure inventory table.

In [6]:
inspect_table_safe('server_inventory_practice')

Column metadata for public.server_inventory_practice


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,public,server_inventory_practice,1,server_id,integer,NO,nextval('server_inventory_practice_server_id_s...
1,public,server_inventory_practice,2,hostname,text,NO,None
2,public,server_inventory_practice,3,service_name,text,NO,None
3,public,server_inventory_practice,4,environment,text,NO,None
4,public,server_inventory_practice,5,region,text,NO,None
5,public,server_inventory_practice,6,cpu_cores,numeric,NO,None
6,public,server_inventory_practice,7,memory_gb,numeric,NO,None


Row count for public.server_inventory_practice


,row_count
0,10


Sample records from public.server_inventory_practice


,server_id,hostname,service_name,environment,region,cpu_cores,memory_gb
0,1,checkout-api-01,checkout-api,prod,us-east-1,8.0,32.0
1,2,checkout-api-02,checkout-api,prod,us-east-1,8.0,32.0
2,3,payment-api-01,payment-api,prod,us-east-1,16.0,64.0
3,4,payment-api-02,payment-api,prod,us-east-1,16.0,64.0
4,5,search-api-01,search-api,prod,us-west-2,8.0,32.0


## 06.3 Count the servers by service

This helps confirm that some services have multiple servers.

That matters because later we can roll up by:

- individual server
- whole service

For example, `checkout-api` may have two servers, so service-level capacity means combining both servers.

In [7]:
sql = """
SELECT
    service_name,
    COUNT(*) AS server_count
FROM server_inventory_practice
GROUP BY service_name
ORDER BY service_name;
"""

run_sql(sql)

,service_name,server_count
0,batch-worker,1
1,checkout-api,2
2,inventory-api,2
3,payment-api,2
4,reporting,1
5,search-api,2


## 06.4 Verify the telemetry sample count

The script generated:

- 24 hours
- 12 samples per hour
- 10 servers

Expected row count:

24 × 12 × 10 = 2,880 rows

This confirms the data was populated correctly.

In [8]:
sql = """
SELECT
    COUNT(*) AS total_samples,
    MIN(sampled_at) AS first_sample,
    MAX(sampled_at) AS last_sample
FROM server_metric_samples_5min;
"""

run_sql(sql)

,total_samples,first_sample,last_sample
0,2880,2026-05-01 00:00:00+00:00,2026-05-01 23:55:00+00:00


## 06.5 Preview raw 5-minute telemetry samples

Now we join the metric table to the inventory table.

The metric table has numeric telemetry.
The inventory table gives the metric rows readable server and service names.

This is an INNER JOIN:

Only metric rows with a matching server_id in the inventory table will appear.

In [9]:
sql = """
SELECT
    m.sampled_at,
    s.hostname,
    s.service_name,
    s.environment,
    s.region,
    m.cpu_utilization_pct,
    m.memory_utilization_pct,
    m.disk_utilization_pct,
    m.p95_latency_ms,
    m.requests_per_min,
    m.error_rate_pct,
    m.actual_cpu_cores,
    m.actual_memory_gb,
    m.cloud_cost_usd,
    m.tags
FROM server_metric_samples_5min m
JOIN server_inventory_practice s
    ON s.server_id = m.server_id
ORDER BY
    m.sampled_at,
    s.service_name,
    s.hostname
LIMIT 30;
"""

run_sql(sql)

,sampled_at,hostname,service_name,environment,region,cpu_utilization_pct,memory_utilization_pct,disk_utilization_pct,p95_latency_ms,requests_per_min,error_rate_pct,actual_cpu_cores,actual_memory_gb,cloud_cost_usd,tags
0,2026-05-01 00:00:00+00:00,batch-worker-01,batch-worker,prod,us-west-2,47.87,72.37,67.78,328,328,0.353,15.32,92.63,0.1857,"{'region': 'us-west-2', 'source': 'synthetic_c..."
1,2026-05-01 00:00:00+00:00,checkout-api-01,checkout-api,prod,us-east-1,64.86,59.95,46.70,286,2677,0.910,5.19,19.18,0.0724,"{'region': 'us-east-1', 'source': 'synthetic_c..."
2,2026-05-01 00:00:00+00:00,checkout-api-02,checkout-api,prod,us-east-1,60.03,50.68,38.89,222,2673,0.849,4.80,16.22,0.0723,"{'region': 'us-east-1', 'source': 'synthetic_c..."
3,2026-05-01 00:00:00+00:00,inventory-api-01,inventory-api,prod,us-east-1,30.84,45.02,46.07,197,1308,0.128,1.23,7.20,0.0359,"{'region': 'us-east-1', 'source': 'synthetic_c..."
4,2026-05-01 00:00:00+00:00,inventory-api-02,inventory-api,prod,us-east-1,32.89,42.06,44.01,220,970,0.617,1.32,6.73,0.0325,"{'region': 'us-east-1', 'source': 'synthetic_c..."
5,2026-05-01 00:00:00+00:00,payment-api-01,payment-api,prod,us-east-1,58.71,59.54,40.83,320,1915,1.030,9.39,38.11,0.1104,"{'region': 'us-east-1', 'source': 'synthetic_c..."
6,2026-05-01 00:00:00+00:00,payment-api-02,payment-api,prod,us-east-1,72.27,69.65,42.45,284,1897,0.553,11.56,44.58,0.1102,"{'region': 'us-east-1', 'source': 'synthetic_c..."
7,2026-05-01 00:00:00+00:00,reporting-01,reporting,prod,us-east-1,35.11,79.81,81.98,357,639,0.276,5.62,102.16,0.1488,"{'region': 'us-east-1', 'source': 'synthetic_c..."
8,2026-05-01 00:00:00+00:00,search-api-01,search-api,prod,us-west-2,54.38,46.85,46.93,141,3274,0.826,4.35,14.99,0.0783,"{'region': 'us-west-2', 'source': 'synthetic_c..."
9,2026-05-01 00:00:00+00:00,search-api-02,search-api,prod,us-west-2,51.96,49.89,39.95,157,2988,0.873,4.16,15.96,0.0755,"{'region': 'us-west-2', 'source': 'synthetic_c..."


## 06.6 Understand the 5-minute grain

This query proves that each server has one row every 5 minutes.

We look at one server only and calculate the time difference between samples.

This uses `LAG()` to compare the current sample time to the previous sample time.

In [10]:
sql = """
SELECT
    s.hostname,
    m.sampled_at,
    LAG(m.sampled_at) OVER (
        PARTITION BY s.hostname
        ORDER BY m.sampled_at
    ) AS previous_sampled_at,
    m.sampled_at
      - LAG(m.sampled_at) OVER (
            PARTITION BY s.hostname
            ORDER BY m.sampled_at
        ) AS time_between_samples
FROM server_metric_samples_5min m
JOIN server_inventory_practice s
    ON s.server_id = m.server_id
WHERE s.hostname = 'checkout-api-01'
ORDER BY m.sampled_at
LIMIT 20;
"""

run_sql(sql)

,hostname,sampled_at,previous_sampled_at,time_between_samples
0,checkout-api-01,2026-05-01 00:00:00+00:00,NaT,NaT
1,checkout-api-01,2026-05-01 00:05:00+00:00,2026-05-01 00:00:00+00:00,0 days 00:05:00
2,checkout-api-01,2026-05-01 00:10:00+00:00,2026-05-01 00:05:00+00:00,0 days 00:05:00
3,checkout-api-01,2026-05-01 00:15:00+00:00,2026-05-01 00:10:00+00:00,0 days 00:05:00
4,checkout-api-01,2026-05-01 00:20:00+00:00,2026-05-01 00:15:00+00:00,0 days 00:05:00
5,checkout-api-01,2026-05-01 00:25:00+00:00,2026-05-01 00:20:00+00:00,0 days 00:05:00
6,checkout-api-01,2026-05-01 00:30:00+00:00,2026-05-01 00:25:00+00:00,0 days 00:05:00
7,checkout-api-01,2026-05-01 00:35:00+00:00,2026-05-01 00:30:00+00:00,0 days 00:05:00
8,checkout-api-01,2026-05-01 00:40:00+00:00,2026-05-01 00:35:00+00:00,0 days 00:05:00
9,checkout-api-01,2026-05-01 00:45:00+00:00,2026-05-01 00:40:00+00:00,0 days 00:05:00


## 06.7 Basic hourly CPU rollup for one service

Now we move from raw 5-minute samples to hourly buckets.

`DATE_TRUNC('hour', sampled_at)` turns timestamps into hour-level buckets.

For each hour and service, we calculate:

- average CPU
- max CPU
- P95 CPU

This is the core capacity rollup pattern.

In [11]:
sql = """
SELECT
    DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
    s.service_name,

    ROUND(AVG(m.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(m.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
        2
    ) AS p95_cpu_pct

FROM server_metric_samples_5min m
JOIN server_inventory_practice s
    ON s.server_id = m.server_id
GROUP BY
    DATE_TRUNC('hour', m.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    p95_cpu_pct DESC;
"""

run_sql(sql)

,sample_hour,service_name,avg_cpu_pct,max_cpu_pct,p95_cpu_pct
0,2026-05-01 00:00:00+00:00,payment-api,63.86,72.83,72.69
1,2026-05-01 00:00:00+00:00,checkout-api,59.47,65.79,65.70
2,2026-05-01 00:00:00+00:00,search-api,52.25,59.39,58.79
3,2026-05-01 00:00:00+00:00,reporting,45.14,52.77,51.28
4,2026-05-01 00:00:00+00:00,batch-worker,39.50,47.87,47.29
...,...,...,...,...,...
139,2026-05-01 23:00:00+00:00,checkout-api,57.60,65.64,64.34
140,2026-05-01 23:00:00+00:00,search-api,50.07,58.58,57.66
141,2026-05-01 23:00:00+00:00,reporting,44.67,50.08,49.65
142,2026-05-01 23:00:00+00:00,batch-worker,38.86,44.07,43.82


## 06.8 Add memory and disk to the hourly rollup

Capacity analysis usually looks at multiple resources together.

CPU alone is not enough.

A server or service may be fine on CPU but risky on memory or disk.

Here we calculate AVG, MAX, and P95 for:

- CPU
- memory
- disk

In [12]:
sql = """
SELECT
    DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
    s.service_name,

    ROUND(AVG(m.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(m.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
        2
    ) AS p95_cpu_pct,

    ROUND(AVG(m.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(m.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.memory_utilization_pct)::NUMERIC,
        2
    ) AS p95_memory_pct,

    ROUND(AVG(m.disk_utilization_pct), 2) AS avg_disk_pct,
    ROUND(MAX(m.disk_utilization_pct), 2) AS max_disk_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.disk_utilization_pct)::NUMERIC,
        2
    ) AS p95_disk_pct

FROM server_metric_samples_5min m
JOIN server_inventory_practice s
    ON s.server_id = m.server_id
GROUP BY
    DATE_TRUNC('hour', m.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    p95_cpu_pct DESC;
"""

run_sql(sql)

,sample_hour,service_name,avg_cpu_pct,max_cpu_pct,p95_cpu_pct,avg_memory_pct,max_memory_pct,p95_memory_pct,avg_disk_pct,max_disk_pct,p95_disk_pct
0,2026-05-01 00:00:00+00:00,payment-api,63.86,72.83,72.69,64.87,69.79,69.64,42.53,47.85,46.90
1,2026-05-01 00:00:00+00:00,checkout-api,59.47,65.79,65.70,56.53,61.02,59.95,42.35,47.92,47.68
2,2026-05-01 00:00:00+00:00,search-api,52.25,59.39,58.79,48.32,53.60,52.62,42.78,47.65,47.33
3,2026-05-01 00:00:00+00:00,reporting,45.14,52.77,51.28,74.84,79.81,79.16,77.77,81.98,81.95
4,2026-05-01 00:00:00+00:00,batch-worker,39.50,47.87,47.29,66.85,72.37,71.98,69.85,74.02,73.56
...,...,...,...,...,...,...,...,...,...,...,...
139,2026-05-01 23:00:00+00:00,checkout-api,57.60,65.64,64.34,56.48,61.70,61.63,43.13,47.54,47.17
140,2026-05-01 23:00:00+00:00,search-api,50.07,58.58,57.66,48.11,53.80,53.16,43.73,47.93,47.72
141,2026-05-01 23:00:00+00:00,reporting,44.67,50.08,49.65,74.22,78.99,78.92,78.43,81.55,81.52
142,2026-05-01 23:00:00+00:00,batch-worker,38.86,44.07,43.82,68.05,73.95,73.04,70.03,74.04,73.95


## 06.9 Add latency, traffic, error rate, and cost

This is closer to a real operational capacity report.

For each service and hour, we summarize:

- utilization
- latency
- traffic
- error rate
- cloud cost

Important latency note:

`p95_latency_ms` is already a 5-minute P95 value.

So this query calculates the P95 of the 5-minute P95 values inside each hour.

That is a useful operational rollup, but it is not the same as true request-level hourly P95.

In [13]:
sql = """
SELECT
    DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
    s.service_name,

    ROUND(AVG(m.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(MAX(m.cpu_utilization_pct), 2) AS max_cpu_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
        2
    ) AS p95_cpu_pct,

    ROUND(AVG(m.memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(MAX(m.memory_utilization_pct), 2) AS max_memory_pct,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.memory_utilization_pct)::NUMERIC,
        2
    ) AS p95_memory_pct,

    ROUND(AVG(m.p95_latency_ms), 2) AS avg_5min_p95_latency_ms,
    MAX(m.p95_latency_ms) AS max_5min_p95_latency_ms,
    ROUND(
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.p95_latency_ms)::NUMERIC,
        2
    ) AS p95_of_5min_p95_latency_ms,

    ROUND(AVG(m.requests_per_min), 0) AS avg_requests_per_min,
    MAX(m.requests_per_min) AS max_requests_per_min,

    ROUND(AVG(m.error_rate_pct), 3) AS avg_error_rate_pct,
    ROUND(MAX(m.error_rate_pct), 3) AS max_error_rate_pct,

    ROUND(SUM(m.cloud_cost_usd), 4) AS hourly_cloud_cost_usd

FROM server_metric_samples_5min m
JOIN server_inventory_practice s
    ON s.server_id = m.server_id
GROUP BY
    DATE_TRUNC('hour', m.sampled_at),
    s.service_name
ORDER BY
    sample_hour,
    p95_cpu_pct DESC;
"""

run_sql(sql)

,sample_hour,service_name,avg_cpu_pct,max_cpu_pct,p95_cpu_pct,avg_memory_pct,max_memory_pct,p95_memory_pct,avg_5min_p95_latency_ms,max_5min_p95_latency_ms,p95_of_5min_p95_latency_ms,avg_requests_per_min,max_requests_per_min,avg_error_rate_pct,max_error_rate_pct,hourly_cloud_cost_usd
0,2026-05-01 00:00:00+00:00,payment-api,63.86,72.83,72.69,64.87,69.79,69.64,285.75,337,335.85,1838.0,2089,0.815,1.090,2.6301
1,2026-05-01 00:00:00+00:00,checkout-api,59.47,65.79,65.70,56.53,61.02,59.95,252.29,298,292.85,2458.0,2677,0.696,0.994,1.6845
2,2026-05-01 00:00:00+00:00,search-api,52.25,59.39,58.79,48.32,53.60,52.62,189.00,256,247.80,3035.0,3283,0.574,0.873,1.8227
3,2026-05-01 00:00:00+00:00,reporting,45.14,52.77,51.28,74.84,79.81,79.16,414.67,466,460.50,479.0,639,0.456,0.725,1.7663
4,2026-05-01 00:00:00+00:00,batch-worker,39.50,47.87,47.29,66.85,72.37,71.98,346.67,402,391.55,412.0,592,0.481,0.800,2.2383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,2026-05-01 23:00:00+00:00,checkout-api,57.60,65.64,64.34,56.48,61.70,61.63,243.33,293,283.10,2499.0,2683,0.636,0.996,1.6944
140,2026-05-01 23:00:00+00:00,search-api,50.07,58.58,57.66,48.11,53.80,53.16,185.00,257,244.25,3061.0,3293,0.567,0.877,1.8290
141,2026-05-01 23:00:00+00:00,reporting,44.67,50.08,49.65,74.22,78.99,78.92,401.50,452,442.10,459.0,744,0.416,0.672,1.7638
142,2026-05-01 23:00:00+00:00,batch-worker,38.86,44.07,43.82,68.05,73.95,73.04,363.33,418,415.80,403.0,554,0.442,0.750,2.2371


## 06.10 Find risky hourly service windows with a CTE

A CTE lets us break the query into two steps:

1. Build the hourly service rollup.
2. Filter that rollup for risky hours.

This is easier to read than writing everything in one giant query.

Risk rules in this example:

- P95 CPU >= 85
- P95 memory >= 85
- P95 of 5-minute P95 latency >= 450 ms
- average error rate >= 1%

In [14]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
        s.service_name,

        ROUND(AVG(m.cpu_utilization_pct), 2) AS avg_cpu_pct,
        ROUND(MAX(m.cpu_utilization_pct), 2) AS max_cpu_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct,

        ROUND(AVG(m.memory_utilization_pct), 2) AS avg_memory_pct,
        ROUND(MAX(m.memory_utilization_pct), 2) AS max_memory_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.memory_utilization_pct)::NUMERIC,
            2
        ) AS p95_memory_pct,

        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.p95_latency_ms)::NUMERIC,
            2
        ) AS p95_of_5min_p95_latency_ms,

        ROUND(AVG(m.error_rate_pct), 3) AS avg_error_rate_pct,
        ROUND(SUM(m.cloud_cost_usd), 4) AS hourly_cloud_cost_usd

    FROM server_metric_samples_5min m
    JOIN server_inventory_practice s
        ON s.server_id = m.server_id
    GROUP BY
        DATE_TRUNC('hour', m.sampled_at),
        s.service_name
)
SELECT *
FROM hourly_service_rollup
WHERE p95_cpu_pct >= 85
   OR p95_memory_pct >= 85
   OR p95_of_5min_p95_latency_ms >= 450
   OR avg_error_rate_pct >= 1.0
ORDER BY
    sample_hour,
    p95_cpu_pct DESC,
    p95_of_5min_p95_latency_ms DESC;
"""

run_sql(sql)

,sample_hour,service_name,avg_cpu_pct,max_cpu_pct,p95_cpu_pct,avg_memory_pct,max_memory_pct,p95_memory_pct,p95_of_5min_p95_latency_ms,avg_error_rate_pct,hourly_cloud_cost_usd
0,2026-05-01 00:00:00+00:00,reporting,45.14,52.77,51.28,74.84,79.81,79.16,460.50,0.456,1.7663
1,2026-05-01 01:00:00+00:00,reporting,45.54,52.99,52.68,74.34,79.97,79.55,468.35,0.432,1.7672
2,2026-05-01 02:00:00+00:00,reporting,42.62,49.29,48.87,73.58,79.11,78.14,463.25,0.458,1.7544
3,2026-05-01 03:00:00+00:00,reporting,43.88,52.70,51.89,72.98,78.78,78.35,454.35,0.417,1.7707
4,2026-05-01 04:00:00+00:00,reporting,44.69,52.73,52.72,75.38,79.71,79.50,468.90,0.437,1.7691
5,2026-05-01 05:00:00+00:00,reporting,43.25,50.09,49.71,73.46,78.76,78.33,469.45,0.420,1.7673
6,2026-05-01 06:00:00+00:00,reporting,43.53,52.31,52.16,72.95,79.64,77.20,459.35,0.498,1.7713
7,2026-05-01 08:00:00+00:00,reporting,43.81,50.81,50.02,73.94,78.43,78.11,461.45,0.456,1.7776
8,2026-05-01 09:00:00+00:00,reporting,43.12,52.63,52.47,74.00,77.53,77.34,464.35,0.311,1.7610
9,2026-05-01 11:00:00+00:00,reporting,44.70,51.77,51.28,71.62,76.41,76.13,466.90,0.317,1.7745


## 06.11 Rank the riskiest service per hour

This uses a window function.

The CTE creates hourly service metrics.
Then `RANK()` ranks services inside each hour.

This answers:

"During each hour, which service had the highest P95 CPU?"

In [15]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
        s.service_name,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.memory_utilization_pct)::NUMERIC,
            2
        ) AS p95_memory_pct,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.p95_latency_ms)::NUMERIC,
            2
        ) AS p95_of_5min_p95_latency_ms
    FROM server_metric_samples_5min m
    JOIN server_inventory_practice s
        ON s.server_id = m.server_id
    GROUP BY
        DATE_TRUNC('hour', m.sampled_at),
        s.service_name
),
ranked AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY sample_hour
            ORDER BY p95_cpu_pct DESC
        ) AS cpu_risk_rank
    FROM hourly_service_rollup
)
SELECT *
FROM ranked
WHERE cpu_risk_rank = 1
ORDER BY sample_hour;
"""

run_sql(sql)

,sample_hour,service_name,p95_cpu_pct,p95_memory_pct,p95_of_5min_p95_latency_ms,cpu_risk_rank
0,2026-05-01 00:00:00+00:00,payment-api,72.69,69.64,335.85,1
1,2026-05-01 01:00:00+00:00,payment-api,71.43,69.74,333.80,1
2,2026-05-01 02:00:00+00:00,payment-api,72.26,69.02,336.55,1
3,2026-05-01 03:00:00+00:00,payment-api,71.34,69.04,333.70,1
4,2026-05-01 04:00:00+00:00,payment-api,71.98,69.09,334.00,1
5,2026-05-01 05:00:00+00:00,payment-api,69.09,69.60,327.85,1
6,2026-05-01 06:00:00+00:00,payment-api,72.57,69.56,327.55,1
7,2026-05-01 07:00:00+00:00,payment-api,71.56,67.31,325.55,1
8,2026-05-01 08:00:00+00:00,payment-api,71.52,67.98,331.05,1
9,2026-05-01 09:00:00+00:00,payment-api,72.61,69.50,330.00,1


## 06.12 Compare current hour to previous hour

This uses `LAG()`.

The goal is to compare each service's current hourly P95 CPU against its previous hour.

This is useful for detecting sudden increases in capacity pressure.

In [16]:
sql = """
WITH hourly_service_rollup AS (
    SELECT
        DATE_TRUNC('hour', m.sampled_at) AS sample_hour,
        s.service_name,
        ROUND(
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY m.cpu_utilization_pct)::NUMERIC,
            2
        ) AS p95_cpu_pct
    FROM server_metric_samples_5min m
    JOIN server_inventory_practice s
        ON s.server_id = m.server_id
    GROUP BY
        DATE_TRUNC('hour', m.sampled_at),
        s.service_name
),
with_previous AS (
    SELECT
        sample_hour,
        service_name,
        p95_cpu_pct,
        LAG(p95_cpu_pct) OVER (
            PARTITION BY service_name
            ORDER BY sample_hour
        ) AS previous_hour_p95_cpu_pct
    FROM hourly_service_rollup
)
SELECT
    sample_hour,
    service_name,
    p95_cpu_pct,
    previous_hour_p95_cpu_pct,
    ROUND(
        p95_cpu_pct - previous_hour_p95_cpu_pct,
        2
    ) AS p95_cpu_change
FROM with_previous
ORDER BY
    service_name,
    sample_hour;
"""

run_sql(sql)

,sample_hour,service_name,p95_cpu_pct,previous_hour_p95_cpu_pct,p95_cpu_change
0,2026-05-01 00:00:00+00:00,batch-worker,47.29,NaN,NaN
1,2026-05-01 01:00:00+00:00,batch-worker,45.24,47.29,-2.05
2,2026-05-01 02:00:00+00:00,batch-worker,45.40,45.24,0.16
3,2026-05-01 03:00:00+00:00,batch-worker,47.55,45.40,2.15
4,2026-05-01 04:00:00+00:00,batch-worker,46.87,47.55,-0.68
...,...,...,...,...,...
139,2026-05-01 19:00:00+00:00,search-api,73.82,76.24,-2.42
140,2026-05-01 20:00:00+00:00,search-api,77.77,73.82,3.95
141,2026-05-01 21:00:00+00:00,search-api,75.01,77.77,-2.76
142,2026-05-01 22:00:00+00:00,search-api,56.17,75.01,-18.84


## 06.13 Query JSONB tags

The `tags` column stores flexible metadata as JSONB.

This is useful because telemetry systems often include labels like:

- service
- environment
- region
- sample_grain
- source

These labels let us filter and group without adding a new physical column for every metadata field.

In [17]:
sql = """
SELECT
    sample_id,
    sampled_at,
    tags,
    tags ->> 'service' AS tag_service,
    tags ->> 'environment' AS tag_environment,
    tags ->> 'region' AS tag_region,
    tags ->> 'sample_grain' AS tag_sample_grain
FROM server_metric_samples_5min
ORDER BY sampled_at
LIMIT 20;
"""

run_sql(sql)

,sample_id,sampled_at,tags,tag_service,tag_environment,tag_region,tag_sample_grain
0,8,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",inventory-api,prod,us-east-1,5_minutes
1,6,2026-05-01 00:00:00+00:00,"{'region': 'us-west-2', 'source': 'synthetic_c...",search-api,prod,us-west-2,5_minutes
2,9,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",reporting,prod,us-east-1,5_minutes
3,4,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",payment-api,prod,us-east-1,5_minutes
4,7,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",inventory-api,prod,us-east-1,5_minutes
5,3,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",payment-api,prod,us-east-1,5_minutes
6,1,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",checkout-api,prod,us-east-1,5_minutes
7,10,2026-05-01 00:00:00+00:00,"{'region': 'us-west-2', 'source': 'synthetic_c...",batch-worker,prod,us-west-2,5_minutes
8,2,2026-05-01 00:00:00+00:00,"{'region': 'us-east-1', 'source': 'synthetic_c...",checkout-api,prod,us-east-1,5_minutes
9,5,2026-05-01 00:00:00+00:00,"{'region': 'us-west-2', 'source': 'synthetic_c...",search-api,prod,us-west-2,5_minutes


## 06.14 Group by a JSONB tag

Here we group by the `region` value inside the JSONB `tags` column.

`tags ->> 'region'` extracts the JSON value as text.

This behaves like a derived column in the query result, even though `region` is not a real column in the metric table.

In [18]:
sql = """
SELECT
    tags ->> 'region' AS region,
    COUNT(*) AS sample_count,
    ROUND(AVG(cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(AVG(memory_utilization_pct), 2) AS avg_memory_pct,
    ROUND(SUM(cloud_cost_usd), 4) AS total_cloud_cost_usd
FROM server_metric_samples_5min
GROUP BY tags ->> 'region'
ORDER BY total_cloud_cost_usd DESC;
"""

run_sql(sql)

,region,sample_count,avg_cpu_pct,avg_memory_pct,total_cloud_cost_usd
0,us-east-1,2016,57.29,58.59,171.0202
1,us-west-2,864,53.38,54.64,99.7599


## 06.15 Final interview explanation

This is the explanation to memorize.

Capacity systems usually do not store every raw event forever. They often collect sampled or pre-aggregated telemetry every few minutes.

In this lab, each server emits one row every 5 minutes.

Then we roll those samples into hourly capacity metrics.

AVG tells me normal usage.
MAX tells me the worst spike.
P95 tells me sustained high pressure while reducing the effect of one-off outliers.

For latency, since `p95_latency_ms` is already a 5-minute P95 value, the hourly latency metric is the P95 of those 5-minute P95 values.

If I had raw request-level latency events, I would calculate the true hourly P95 from those raw events.